# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

# Simple Chain

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))
print(chain.invoke(input= {'city': '강원도'}))

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 강원도 하면 가장 먼저 떠오르는 대표 농산물
- **옥수수**: 특히 찰옥수수가 유명함
- **메밀**: 봉평 메밀과 메밀국수·메밀전병 등
- **황태**: 인제·평창 일대의 겨울철 건조 명태
- **오징어**: 동해안 지역의 대표 수산물
- **고랭지 배추·무**: 평창, 강릉, 태백 등 고랭지 채소
- **한우**: 횡성한우가 특히 유명함
- **송이버섯·산나물**: 양양 송이, 곰취·더덕 등
- **초당두부**: 강릉의 대표 먹거리
- **닭갈비**: 춘천을 대표하는 향토 음식

지역에 따라 특산물이 조금씩 다르며, 농산물뿐 아니라 황태·오징어 같은 수산물과 향토 음식도 강원도의 대표 특산물로 꼽힙니다.
강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창·강릉·횡성 등에서 많이 생산되며, 수미감자와 감자떡이 유명합니다.
- **옥수수**: 홍천·정선·원주 등의 찰옥수수가 대표적입니다.
- **황태**: 인제 용대리 황태가 특히 유명합니다.
- **오징어**: 동해안 지역의 건오징어와 반건조 오징어가 대표적입니다.
- **곤드레**: 정선 곤드레나물과 곤드레밥이 유명합니다.
- **메밀**: 평창·봉평의 메밀과 메밀국수, 메밀전병이 잘 알려져 있습니다.
- **한우**: 횡성한우가 대표적인 고급 축산물입니다.
- **더덕·산나물·송이버섯**: 산간 지역에서 많이 생산됩니다.
- **닭갈비와 막국수**: 춘천을 대표하는 향토 음식입니다.


# Sequential Chain

In [10]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm 

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""


print(chain1.invoke(eng_text))


chain2 = prompt2 | llm | output_parser
kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""
print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 극복하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.\n\n이를 위해 먼저 문서 로더(document loader)를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 97, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHz5RronQuY2sEKp2TYxagek0pou', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0408d-9418-7681-944c-12dde79ace1a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={

In [11]:
chain = chain1 | chain2

print(chain.invoke({'eng_text': eng_text}))

LLM은 외부 문서나 이메일 등 특정 데이터에 직접 접근하지 못하는 맥락 정보 부족의 한계가 있습니다. 이를 보완하려면 외부 데이터를 불러와 LLM에 제공해야 하며, LangChain은 PDF·이메일·웹사이트·YouTube 등 다양한 자료를 처리할 수 있는 문서 로더를 제공합니다.


# Conditional Chain

In [12]:
from langchain_core.runnables import RunnableBranch

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단게적인 풀이를 수식(LaTex)와 함께 작성해주세요. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감석적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')
    return '계산' in question or 'calc' in question

branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

단계적으로 계산하면 다음과 같습니다.

\[
125 \times 3 = 375
\]

따라서,

\[
375 + 50 = 425
\]

정답은

\[
\boxed{425}
\]


### Memory Chain

'RunnableWithMessageHistory'를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [13]:
from langchain_core.chat_history import BaseChatMessageHistory              # LangChain 대화기록 메모리 저장용 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage    # 메시지 타입들
from pydantic import BaseModel, Field       # Pydantic 모델(검증/ 기본값 생성) 도구
from typing import List     # 타입 힌트(List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factory = list) : 인스턴스마다 독립적인 messages list 를 구성
    messages : List[BaseMessage] = Field(default_factory= list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)      # 전달받은 메시지들을 기존 리스트 뒤에 추가
    
    def clear(self) -> None:
        self.messages = []  # 저장된 메시지들을 초기화
        
store = {}  # {session_id : 히스토리 객체(InMemoryHistory)} 저장소

def get_by_session_id(session_id: str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()   # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]        # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')   # 세션 ID '1'의 히스토리 가져오기(없으면 메모리 공간 생성)
history1.add_messages([AIMessage(content= '반갑습니다. Capybara님!')])      # AI 메시지 추가
history1.add_messages([HumanMessage(content= '그래~ 나 Cap이야~ 만나서 반갑다!!')]) # 유저 메시지 추가
print(f'{history1 = }')     


history2 = get_by_session_id('2')
print(f'{history2 = }')


history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder      # 채팅 프롬프트 템플릿/ 히스토리 자리 표시자
from langchain_core.runnables import RunnableWithMessageHistory         # 실행 시 히스토리를 붙여주는 Runnable

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name= 'history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,              # sessionID 로 히스토리 객체 가져오는 함수 참조
    input_messages_key= 'question', # 입력 dict 에서 question 키의 값은 사용자 메시지
    history_messages_key= 'history' # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config = {       # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100' # 어떤 세션 히스토리 사용할지
    }
}
)

c:\Users\MoonSungHo\skn34-st\study\DL\DL_venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='소라는 고양이 4마리, 민수는 강아지 3마리를 키우고 있으므로 두 사람이 키우는 반려동물은 모두 **7마리**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 86, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 47, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI64VKQUMY38R7ef2eHNpcJjEVU2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04094-2cd7-7fd3-9998-5614ea39b990-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 86, 'output_tokens': 99, 'total_tokens': 185, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'aud

In [16]:
chain_with_history.invoke({
    'domain': 'math',
    'question': '소라는 고양이를 4마리 키우고 있습니다.'
}, config = {       # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100' # 어떤 세션 히스토리 사용할지
    }
})

AIMessage(content='네, 소라는 고양이 **4마리**를 키우고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 153, 'total_tokens': 211, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 31, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI66miYRkSVQMZrbKwjM49SIzeJy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04094-3648-7e81-8c6e-202d045c6aa9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 58, 'total_tokens': 211, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 31}})

In [17]:
store   # 현재 메모리에 저장된 세션별 대화 히스토리

{'1': InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})]),
 '2': InMemoryHistory(messages=[]),
 '100': InMemoryHistory(messages=[HumanMessage(content='소라는 고양이를 4마리 키우고 있습니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='소라는 고양이 4마리를 키우고 있군요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 37, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 33, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmp

### ChatMessageHistory

In [22]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

def get_by_session_id(session_id: str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()   # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]        # 해당 세션의 히스토리 객체 반환


prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name= 'history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

chain_with_history.invoke({
    'domain': '건강전문가',
    'question': '감기에 걸려있는데, 저녁에 잘때 선풍기를 안틀고자면 너무 덥고, 선풍기를 틀고 자면 아침에 목이 너무 아파'
}, config = {       # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '300' # 어떤 세션 히스토리 사용할지
    }
}
)

AIMessage(content='감기 중에는 **선풍기 바람이 목과 코를 직접 말려서** 아침에 통증이 더 심해질 수 있습니다. 선풍기 자체가 감기를 악화시키는 것은 아니지만, 건조함·입 벌리고 자기·먼지가 증상을 자극할 수 있어요.\n\n다음처럼 해보세요.\n\n- 선풍기를 **몸이나 얼굴에 직접 향하지 말고**, 벽이나 천장을 향해 간접적으로 틀기  \n- 약풍 또는 회전 기능을 사용하고, **잠들기 전 타이머** 설정  \n- 방이 너무 덥지 않게 하되, 가능하면 **24~26℃ 정도**로 유지  \n- 물을 자주 마시고, 자기 전 따뜻한 물이나 차를 조금 마시기  \n- 가습기를 사용한다면 습도 **40~60%**를 목표로 하고 매일 물을 갈고 세척하기  \n- 코막힘이 있으면 생리식염수 스프레이나 코 세척을 고려하고, 코로 숨 쉬도록 하기  \n- 선풍기와 침구의 먼지를 청소하기\n\n목 통증이 심하면 일반적으로 아세트아미노펜이나 이부프로펜이 도움이 될 수 있지만, 간질환·위궤양·신장질환, 임신 중이거나 복용 중인 약이 있으면 먼저 약사나 의사에게 확인하세요.\n\n**고열이 3일 이상 지속되거나, 숨쉬기 어렵고 침도 삼키기 힘들거나, 한쪽 목만 심하게 붓거나, 증상이 1~2주 이상 지속되면** 진료를 받으세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 456, 'prompt_tokens': 67, 'total_tokens': 523, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 79, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_t

In [23]:
chain_with_history.invoke({
    'domain': '건강전문가',
    'question': '코가 막혀있어서 입으로 숨쉴수밖에 없잖아'
}, config = {       # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '300' # 어떤 세션 히스토리 사용할지
    }
}
)

AIMessage(content='맞아요. 코가 막히면 입으로 숨 쉬게 되어 **목이 마르고 아침에 통증이 심해질 수 있습니다.** 선풍기를 끄기보다, 코막힘을 먼저 줄이고 바람을 간접적으로 쓰는 게 좋습니다.\n\n- 자기 전 **생리식염수 스프레이**를 사용하거나 코 세척하기  \n  - 코 세척은 반드시 **멸균수·끓였다 식힌 물·정수된 물**을 사용하세요.\n- 따뜻한 샤워나 수증기로 코를 촉촉하게 하기\n- 베개를 조금 높여 자기\n- 방은 서늘하게 유지하되 선풍기는 **얼굴에서 멀리, 벽 방향·약풍·타이머**로 설정\n- 침대 옆에 물을 두고, 입술·목이 마르면 조금씩 수분 섭취\n\n코막힘이 심하면 약국의 **혈관수축성 코 스프레이(예: 옥시메타졸린·자일로메타졸린)**가 빠르게 도움이 될 수 있지만, **3일 이내만** 사용해야 반동성 코막힘을 피할 수 있습니다. 고혈압·심장질환·갑상선질환·전립선 질환이 있거나 임신 중이면 약사와 먼저 상의하세요.\n\n코막힘이 10일 이상 지속되거나, 고열·심한 얼굴 통증·누런 콧물 악화·숨쉬기 어려움이 있으면 진료를 받으세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 426, 'prompt_tokens': 463, 'total_tokens': 889, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 72, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_

In [21]:
print(store['200'])

Human: 요즘 갑자기 더워져서 짜증나는데? 나 성격좋은데? 왜이러지?
AI: 그럴 수 있어요. **더위 때문에 짜증이 늘었다고 해서 성격이 나빠진 건 아니에요.** 몸이 열을 식히느라 에너지를 더 쓰고, 수면의 질이 떨어지거나 탈수·피로가 쌓이면 감정을 조절하는 여유가 줄어들 수 있습니다. 즉, 성격보다는 **몸의 스트레스 반응**에 가까워요.

도움이 될 만한 건:

- 물을 자주 마시고, 시원한 환경에서 잠자기  
- 더운 시간대엔 일정과 자극을 조금 줄이기  
- 짜증이 올라올 때 바로 반응하기보다 “지금 더워서 예민해졌구나” 하고 잠깐 멈추기  
- 가벼운 샤워, 목·손목 식히기, 느린 호흡하기  
- 잠을 충분히 보충하기

다만 더위와 상관없이 짜증·불안·우울이 오래 지속되거나, 두근거림·어지럼·심한 무기력까지 있다면 건강 상태도 확인해보는 게 좋아요. 지금은 특히 어떤 상황에서 짜증이 가장 많이 나나요?
Human: 짜증이 나서 다른사람을 못살게 굴고있는데 성격이 나빠진게 아니야?
AI: 짜증이 나는 감정 자체가 성격이 나쁘다는 뜻은 아니지만, **그 짜증으로 다른 사람을 반복해서 괴롭히거나 상처 주는 행동은 분명히 조절하고 책임져야 해요.** 더위가 원인일 수는 있어도 면죄부는 아니에요. 성격이 “나빠졌다”기보다, 지금 몸과 감정의 여유가 줄어 행동이 거칠어지고 있는 상태일 수 있습니다.

우선 이렇게 해보세요.

1. **대화를 즉시 중단하고 거리 두기**  
   “지금 예민해서 더 말하면 상처 줄 것 같아. 20분 뒤에 이야기할게.”
2. **몸부터 식히기**  
   물 마시기, 시원한 곳으로 이동, 세수나 샤워, 천천히 호흡하기.
3. **상대에게 구체적으로 사과하기**  
   “더워서 그랬어”로 끝내기보다, “내가 아까 큰소리 내고 몰아붙인 건 잘못이었어. 미안해. 다음엔 화가 나면 먼저 자리를 피할게.”
4. **반복되면 원인 점검하기**  
   수면 부족, 탈수, 스트레스, 카페인·술 등이 짜증을 키울 수 있어요.

혹시 지금

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리